# Groundwater data cleaning

Prepare groundwater nitrate measurements for modeling within the HUC8 watershed.


## 1. Imports

In [1]:
# Imports
import pandas as pd
import geopandas as gpd
from pathlib import Path


## 2. Load groundwater data

In [2]:
# Load groundwater data
groundwater = pd.read_csv(
    "groundwaterdata.csv",
    dtype={"src_samp_collection_time": "string"},
    low_memory=False
)

groundwater.head()


,_id,gm_county_name,gm_dataset_name,gm_well_category,gm_data_source,gm_well_id,gm_chemical_vvl,gm_chemical_name,gm_result_modifier,gm_result,...,src_analysis_date,src_analytical_method,src_lab_note,src_latitude,src_longitude,src_datum,src_well_depth_ft,src_top_depth_of_screen_ft,src_bottom_depth_of_screen_ft,src_well_category
0,1,SAN LUIS OBISPO,WB_ILRP,Domestic,GeoTracker,AGL020003373-ROLLING DOM 1,TDS,Total Dissolved Solids,=,350.00000,...,2013-07-02,A2540C,NaN,35.738160,-120.633550,NAD83,NaN,NaN,NaN,Domestic/Private Drinking Water Well
1,2,SANTA CRUZ,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020002084-AG_WELL_S,B,Boron,ND,0.00053,...,2012-09-21,E200.7,NaN,37.048478,-122.226403,NAD83,NaN,NaN,NaN,Agriculture/irrigation well
2,3,SANTA BARBARA,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020005540-PRIMARY,SC,Specific Conductivity,=,1400.00000,...,2012-11-16,2510B,NaN,34.934444,-120.470556,NAD83,NaN,NaN,NaN,Agriculture/irrigation well
3,4,SANTA BARBARA,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020027102-RIVER RCH,NO3N,Nitrate as N,=,39.60000,...,2017-06-24,E300.0,NaN,34.969640,-120.481420,NAD83,NaN,NaN,NaN,Agriculture/irrigation well
4,5,MONTEREY,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020001277-BASSI_AG2,CL,Chloride,=,63.89000,...,2017-12-21,E300.0,NaN,36.525190,-121.457610,NAD83,NaN,NaN,NaN,Agriculture/irrigation well


## 3. Load the HUC8 watershed boundary

In [3]:
# Load HUC8 watershed boundary
gdb = Path("NHD_H_18060005_HU8_GDB") / "NHD_H_18060005_HU8_GDB.gdb"

huc8 = gpd.read_file(
    gdb,
    layer="WBDHU8"
)

huc8.head()


,tnmid,metasourceid,sourcedatadesc,sourceoriginator,sourcefeatureid,loaddate,referencegnis_ids,areaacres,areasqkm,states,huc8,name,shape_Length,shape_Area,geometry
0,{2EB2E16D-AA63-432F-9DC1-1820721507ED},None,None,None,None,2012-06-11 07:54:57+00:00,None,2130628.5,8622.36,CA,18060005,Salinas,7.476846,0.862215,"MULTIPOLYGON (((-121.79903 36.74909, -121.7986..."


## 4. Convert groundwater records to spatial points

In [4]:
# Convert groundwater coordinates to geographic points
groundwater_gdf = gpd.GeoDataFrame(
    groundwater,
    geometry=gpd.points_from_xy(
        groundwater["gm_longitude"],
        groundwater["gm_latitude"]
    ),
    crs="EPSG:4326"
)

groundwater_gdf.head()


,_id,gm_county_name,gm_dataset_name,gm_well_category,gm_data_source,gm_well_id,gm_chemical_vvl,gm_chemical_name,gm_result_modifier,gm_result,...,src_analytical_method,src_lab_note,src_latitude,src_longitude,src_datum,src_well_depth_ft,src_top_depth_of_screen_ft,src_bottom_depth_of_screen_ft,src_well_category,geometry
0,1,SAN LUIS OBISPO,WB_ILRP,Domestic,GeoTracker,AGL020003373-ROLLING DOM 1,TDS,Total Dissolved Solids,=,350.00000,...,A2540C,NaN,35.738160,-120.633550,NAD83,NaN,NaN,NaN,Domestic/Private Drinking Water Well,POINT (-120.63355 35.73816)
1,2,SANTA CRUZ,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020002084-AG_WELL_S,B,Boron,ND,0.00053,...,E200.7,NaN,37.048478,-122.226403,NAD83,NaN,NaN,NaN,Agriculture/irrigation well,POINT (-122.2264 37.04848)
2,3,SANTA BARBARA,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020005540-PRIMARY,SC,Specific Conductivity,=,1400.00000,...,2510B,NaN,34.934444,-120.470556,NAD83,NaN,NaN,NaN,Agriculture/irrigation well,POINT (-120.47056 34.93444)
3,4,SANTA BARBARA,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020027102-RIVER RCH,NO3N,Nitrate as N,=,39.60000,...,E300.0,NaN,34.969640,-120.481420,NAD83,NaN,NaN,NaN,Agriculture/irrigation well,POINT (-120.48142 34.96964)
4,5,MONTEREY,WB_ILRP,Irrigation / Industrial,GeoTracker,AGL020001277-BASSI_AG2,CL,Chloride,=,63.89000,...,E300.0,NaN,36.525190,-121.457610,NAD83,NaN,NaN,NaN,Agriculture/irrigation well,POINT (-121.45761 36.52519)


## 5. Match coordinate reference systems

In [5]:
# Check coordinate reference systems
print(groundwater_gdf.crs)
print(huc8.crs)


EPSG:4326
EPSG:4269


In [6]:
# Reproject HUC8 to match groundwater CRS
huc8 = huc8.to_crs(groundwater_gdf.crs)

print(groundwater_gdf.crs)
print(huc8.crs)


EPSG:4326
EPSG:4326


## 6. Clip groundwater data to the HUC8 watershed

In [7]:
# Keep only groundwater points inside the HUC8 watershed
groundwater_huc8 = gpd.clip(
    groundwater_gdf,
    huc8
)

groundwater_huc8.shape


(94859, 40)

In [8]:
# Check counties represented inside the HUC8
groundwater_huc8["gm_county_name"].value_counts()


gm_county_name
MONTEREY           61058
SAN LUIS OBISPO    33769
SAN BENITO            32
Name: count, dtype: int64

## 7. Remove exact duplicate rows

In [9]:
# Remove exact duplicate rows
groundwater_huc8 = groundwater_huc8.drop_duplicates()

groundwater_huc8.shape


(94859, 40)

## 8. Remove censored measurements

In [10]:
# Flag censored measurements
groundwater_huc8["is_censored"] = (
    groundwater_huc8["gm_result_modifier"] != "="
)

groundwater_huc8["is_censored"].value_counts()


is_censored
False    87816
True      7043
Name: count, dtype: int64

In [11]:
# Keep only uncensored measurements
groundwater_clean = groundwater_huc8[
    groundwater_huc8["is_censored"] == False
].copy()

groundwater_clean.shape


(87816, 41)

## 9. Standardize sample date and time

In [12]:
# Convert sample date to datetime
groundwater_clean["gm_samp_collection_date"] = pd.to_datetime(
    groundwater_clean["gm_samp_collection_date"]
)

# Standardize sampling time as four-character text
groundwater_clean["src_samp_collection_time"] = (
    groundwater_clean["src_samp_collection_time"]
    .astype("string")
    .str.zfill(4)
)

groundwater_clean[
    ["gm_samp_collection_date", "src_samp_collection_time"]
].head()


,gm_samp_collection_date,src_samp_collection_time
128973,2017-04-21,0943
280116,2025-03-26,1345
5692,2017-12-05,1005
232056,2017-12-05,1005
195945,2023-06-13,1020


## 10. Keep only nitrate-related measurements

In [13]:
# Check available chemical measurements
groundwater_clean["gm_chemical_name"].value_counts().head(30)


gm_chemical_name
pH                                    11450
Specific Conductivity                 11220
Total Dissolved Solids                 8957
Temperature                            6805
Nitrate+Nitrite                        6574
Nitrate as N                           5335
Sodium                                 4977
Magnesium                              4963
Calcium                                4963
Sulfate                                4951
Chloride                               4929
Potassium                              4894
Alkalinity, total                      4628
bicarbonate HCO3                        749
Nitrite as N                            690
Boron                                   525
Dissolved Oxygen (DO)                   444
Iron                                    300
Manganese                               202
1,2,3-Trichloropropane (1,2,3 TCP)      159
Hardness                                 63
Bicarbonate Alkalinity                   33
carbonate CO3  

In [14]:
# Keep only nitrate-related measurements
groundwater_nitrate = groundwater_clean[
    groundwater_clean["gm_chemical_name"].isin(
        ["Nitrate as N", "Nitrate+Nitrite"]
    )
].copy()

groundwater_nitrate.shape


(11909, 41)

## 11. Check repeated nitrate measurements

In [15]:
# Check repeated nitrate measurements from the same sample
duplicate_measurements = groundwater_nitrate.duplicated(
    subset=[
        "gm_well_id",
        "gm_samp_collection_date",
        "src_samp_collection_time",
        "gm_chemical_name"
    ],
    keep=False
)

duplicate_measurements.sum()


np.int64(1654)

In [16]:
# Inspect repeated nitrate measurements
groundwater_nitrate[
    duplicate_measurements
][
    [
        "gm_well_id",
        "gm_samp_collection_date",
        "src_samp_collection_time",
        "gm_chemical_name",
        "gm_result",
        "gm_reporting_limit"
    ]
].sort_values(
    [
        "gm_well_id",
        "gm_samp_collection_date",
        "src_samp_collection_time",
        "gm_chemical_name"
    ]
).head(20)


,gm_well_id,gm_samp_collection_date,src_samp_collection_time,gm_chemical_name,gm_result,gm_reporting_limit
162803,AGC100000001-CCGC_0001,2013-10-24,1715,Nitrate as N,4.7,0.1
52363,AGC100000001-CCGC_0001,2013-10-24,1715,Nitrate as N,4.7,1.0
302340,AGC100000001-CCGC_0002,2013-10-24,1300,Nitrate as N,21.8,0.1
216575,AGC100000001-CCGC_0002,2013-10-24,1300,Nitrate as N,22.0,1.0
289992,AGC100000001-CCGC_0003,2013-10-24,1615,Nitrate as N,66.9,1.0
133903,AGC100000001-CCGC_0003,2013-10-24,1615,Nitrate as N,66.8,0.1
310112,AGC100000001-CCGC_0005,2013-10-21,1110,Nitrate as N,7.2,1.0
261416,AGC100000001-CCGC_0005,2013-10-21,1110,Nitrate as N,7.3,0.1
137214,AGC100000001-CCGC_0006,2013-10-25,1340,Nitrate as N,53.5,1.0
63918,AGC100000001-CCGC_0006,2013-10-25,1340,Nitrate as N,53.6,0.1


In [17]:
duplicate_measurements.sum()

np.int64(1654)

In [18]:
groundwater_nitrate[
    duplicate_measurements
][
    [
        "gm_well_id",
        "gm_samp_collection_date",
        "src_samp_collection_time",
        "gm_chemical_name",
        "gm_result",
        "gm_reporting_limit"
    ]
].head(20)

,gm_well_id,gm_samp_collection_date,src_samp_collection_time,gm_chemical_name,gm_result,gm_reporting_limit
99768,AGL020000648-MAIN_D/I,2014-06-02,1300,Nitrate as N,2.5,1.0
145450,AGL020000648-MAIN_D/I,2014-01-07,1340,Nitrate as N,2.3,1.0
115054,AGL020000648-MAIN_D/I,2014-06-02,1300,Nitrate as N,2.4,0.1
113116,AGL020000648-MAIN_D/I,2014-01-07,1340,Nitrate as N,2.2,0.1
35055,AGL020002700-CARIAG_D/I,2014-01-06,0930,Nitrate as N,2.9,1.0
96822,AGL020007899-DOM/IRR,2014-06-02,0920,Nitrate as N,2.5,1.0
84366,AGL020007899-DOM/IRR,2014-01-30,0850,Nitrate as N,2.8,0.1
60668,AGL020007899-DOM/IRR,2014-01-30,0850,Nitrate as N,2.7,1.0
119310,AGL020007899-DOM/IRR,2014-06-02,0920,Nitrate as N,2.5,0.1
187541,AGL020002700-CARIAG_D/I,2014-06-02,1000,Nitrate as N,0.6,0.1


In [19]:
# Average replicate nitrate measurements from the same sample
groundwater_nitrate_clean = (
    groundwater_nitrate
    .groupby(
        [
            "gm_well_id",
            "gm_well_category",
            "gm_samp_collection_date",
            "src_samp_collection_time",
            "gm_chemical_name",
            "gm_latitude",
            "gm_longitude"
        ],
        as_index=False
    )
    .agg(
        gm_result=("gm_result", "mean"),
        gm_reporting_limit=("gm_reporting_limit", "min")
    )
)

In [20]:
groundwater_nitrate_clean.shape

(11079, 9)

In [21]:
groundwater_nitrate_clean.duplicated(
    subset=[
        "gm_well_id",
        "gm_samp_collection_date",
        "src_samp_collection_time",
        "gm_chemical_name"
    ]
).sum()

np.int64(0)

In [22]:
# Create one nitrate concentration per groundwater sample
groundwater_final = (
    groundwater_nitrate_clean
    .groupby(
        [
            "gm_well_id",
            "gm_well_category",
            "gm_samp_collection_date",
            "src_samp_collection_time",
            "gm_latitude",
            "gm_longitude"
        ],
        as_index=False
    )
    .agg(
        nitrate_mg_L=("gm_result", "mean")
    )
)

groundwater_final.shape

(10665, 7)

In [23]:
# Final check of the target variable
groundwater_final["nitrate_mg_L"].describe()

count    10665.000000
mean        13.123557
std         21.049301
min          0.000000
25%          1.500000
50%          4.200000
75%         15.500000
max        230.000000
Name: nitrate_mg_L, dtype: float64

In [24]:
# Final check of the target variable
groundwater_final["nitrate_mg_L"].describe()

count    10665.000000
mean        13.123557
std         21.049301
min          0.000000
25%          1.500000
50%          4.200000
75%         15.500000
max        230.000000
Name: nitrate_mg_L, dtype: float64

In [25]:
# Check for missing values
groundwater_final.isna().sum()

gm_well_id                  0
gm_well_category            0
gm_samp_collection_date     0
src_samp_collection_time    0
gm_latitude                 0
gm_longitude                0
nitrate_mg_L                0
dtype: int64

In [26]:
# Save cleaned groundwater target dataset
groundwater_final.to_csv(
    "groundwater_nitrate_clean.csv",
    index=False
)

In [27]:
# Convert final groundwater table back to a GeoDataFrame
groundwater_final_gdf = gpd.GeoDataFrame(
    groundwater_final,
    geometry=gpd.points_from_xy(
        groundwater_final["gm_longitude"],
        groundwater_final["gm_latitude"]
    ),
    crs="EPSG:4326"
)

# Save spatial version
groundwater_final_gdf.to_file(
    "groundwater_nitrate_clean.gpkg",
    layer="groundwater_nitrate",
    driver="GPKG"
)